# 04 - Apprentissage semi-supervisé & comparaison
**Branche** : `feature/04-semi-supervised`

**Definition of done** : le meilleur modèle semi-supervisé atteint un rappel « cancer » ≥ celui du supervisé seul (moyenne sur 5 répétitions), avec un F1 au moins égal et un AUC-ROC ≥ 0,85 - sinon on conclut honnêtement que le semi-supervisé n'apporte rien ici.

**Métrique principale** : rappel (sensibilité) sur « cancer » - un faux négatif (tumeur manquée) est l'erreur la plus grave. Compléments : F1/F2, précision, spécificité, AUC-ROC/PR. Accuracy seule = trompeuse (test petit, classes éventuellement déséquilibrées).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from brainscanai.config import load_config, resolve_path
from brainscanai.utils import set_seed

cfg = load_config()
set_seed(cfg["seed"])
PROC, FEAT = resolve_path(cfg, "processed", create=True), resolve_path(cfg, "features", create=True)
FIG, TAB = resolve_path(cfg, "figures", create=True), resolve_path(cfg, "tables", create=True)
from brainscanai.features.io import load_features
from brainscanai.data.split import assign_roles
from brainscanai.semi_supervised.embedding_experiment import run_embedding_experiment
from brainscanai.evaluation.metrics import *
from brainscanai.evaluation.plots import *

In [ ]:
manifest = pd.read_csv(PROC / 'manifest_clean.csv')
X, df = load_features(FEAT / 'embeddings.npz', manifest)
d = assign_roles(df, cfg['data']['test_size'], cfg['seed'])
d.groupby(['role', 'label_name']).size().unstack(fill_value=0)

## A. Méthodes sur embeddings (rapide)
Supervisé, labels faibles, faible+fort, Self-training (pseudo-labelling), Label Spreading, Label Propagation - 5 répétitions (splits différents).

In [ ]:
summary, probs = run_embedding_experiment(df, X, cfg)
summary.to_csv(TAB / 'semi_supervised_embeddings.csv', index=False)
summary[['approach', 'recall_cancer_mean', 'recall_cancer_std', 'precision_cancer_mean', 'f1_mean', 'roc_auc_mean', 'accuracy_mean']].round(3)

In [ ]:
fig = plot_metric_comparison(summary, title='Embeddings : supervisé vs semi-supervisé'); save_fig(fig, 'compare_embeddings', FIG); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
curves = {k: roc_points(probs['y_test'], v) for k, v in probs.items() if k != 'y_test'}
plot_roc_curves(curves); plt.show()
best = summary.sort_values('recall_cancer_mean').iloc[-1].approach
plot_confusion_matrix(confusion_matrix(probs['y_test'], probs[best] >= .5), cfg['data']['classes'], best); plt.show()

## B. Protocole CNN (mission) : faible → fort vs fort seul (+ pseudo-labels du modèle)
Quatre bras comparés sur les **mêmes** splits : **A** labels faibles seuls ; **B** faible → fort (semi-supervisé) ; **C** fort seul (supervisé) ; **D** pseudo-labelling par le modèle (C prédit les images sans label, on garde les prédictions très confiantes, puis réentraînement conjoint). B et C utilisent les **mêmes hyperparamètres** pour être comparables.
Demande un GPU idéalement. Réduire `n_repeats` et les epochs pour un essai.

In [ ]:
from brainscanai.semi_supervised.cnn_experiment import run_cnn_experiment
cnn_summary, cnn_probs, cnn_diag = run_cnn_experiment(df, X, cfg, n_repeats=2)   # passer à 5 pour le rendu final
cnn_summary.to_csv(TAB / 'semi_supervised_cnn.csv', index=False)
cnn_diag.to_csv(TAB / 'cnn_diagnostics.csv', index=False)
display(cnn_diag.round(3))   # weak_label_acc_on_test = qualité des labels faibles ; n_pseudo_kept = pseudo-labels retenus pour D
cnn_summary[['approach', 'recall_cancer_mean', 'f1_mean', 'roc_auc_mean', 'accuracy_mean']].round(3)

In [ ]:
fig = plot_metric_comparison(cnn_summary, title='CNN : supervisé vs semi-supervisé'); save_fig(fig, 'compare_cnn', FIG); plt.show()

## C. Incertitude (test petit) & seuil orienté rappel
Avec ~30 images de test, un IC bootstrap est indispensable ; le seuil de décision doit être calibré sur la **validation**, pas sur le test.

In [ ]:
y = cnn_probs['y_test']
for k in [a for a in ['A', 'B', 'C', 'D'] if a in cnn_probs]:
    lo, hi = bootstrap_ci(y, cnn_probs[k], 'recall_cancer')
    print(k, 'rappel IC95% = [%.2f, %.2f]' % (lo, hi))

## D. Hyperparamètres
Un script d'ablation est fourni : `python scripts/run_ablation.py --keep 0.3 0.5 0.8 --repeats 3` (qualité vs quantité des labels faibles, mêmes splits). Grille complète à explorer (validation croisée sur `strong_train`) : `lr ∈ {3e-5, 1e-4, 3e-4}`, `epochs ∈ {10, 15, 25}`, `keep_fraction ∈ {0.6, 0.8, 1.0}`, `self_training_threshold ∈ {0.8, 0.9, 0.95}`, `n_neighbors ∈ {5, 7, 11}`. Reporter les résultats dans `reports/tables/`.

## E. Conclusion & recommandations
_À compléter avec les vrais chiffres : le semi-supervisé bat-il le supervisé ? De combien (± écart-type) ? Les labels faibles sont-ils assez propres ? Lien avec le scale-up (`reports/tables/scale_up_budget.csv`)._